# AgriSmart AI — Export trained model to ONNX

Converts your trained `agrismart_resnet50.keras` into a small ONNX model that runs **both** on the Render API (no TensorFlow, fits 512 MB) **and** inside the browser (offline PWA).

It also produces the evidence judges ask for: Keras-vs-ONNX parity, Grad-CAM similarity, per-class precision/recall and a confusion matrix, all on the **same held-out test split** used in training (`random_state=42`).

**Steps:** Runtime → Change runtime type → **T4 GPU**, then run the cells in order.


In [ ]:
!git clone https://github.com/Anshum25/agrismart-ai.git
%cd agrismart-ai

In [ ]:
!pip install -q tf2onnx onnx onnxruntime kaggle opencv-python-headless scikit-learn
# If tf2onnx later fails with a NumPy 2 error, run:  !pip install -q "numpy<2"  then Runtime -> Restart and re-run from the %cd cell.

## 1. Provide the trained model
Either mount Google Drive (if training saved there) or upload `agrismart_resnet50.keras` (+ `class_labels.json` if you have it).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p model/weights
# Adjust these paths to where your files are:
!cp "/content/drive/MyDrive/AgriSmartAI/weights/agrismart_resnet50.keras" model/weights/
!cp "/content/drive/MyDrive/AgriSmartAI/weights/class_labels.json" model/weights/ || echo "no class_labels.json - default alphabetical labels will be used"
!ls -lh model/weights

## 2. Download PlantVillage (needed for calibration, parity and samples)
Upload your `kaggle.json` when prompted.

In [ ]:
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p data/plantvillage --unzip -q
!ls data/plantvillage

## 3. Export + quantize + verify

In [ ]:
!python model/export_onnx.py     --keras model/weights/agrismart_resnet50.keras     --data-dir "data/plantvillage/plantvillage dataset/color"     --full-eval

In [ ]:
import json
print(json.dumps(json.load(open('model/weights/export/parity_report.json')), indent=2))
!ls -lh model/weights/export/web/model model/weights/export/web/samples

## 4. Download and commit
Unzip `agrismart_export.zip` and copy its `model/` and `samples/` folders into `frontend/public/` in the repo, then commit and push.

Targets: `int8_top1_agreement ≥ 0.99`, accuracy drop ≤ 1%, `gradcam_cosine_similarity_mean > 0.99` (int8 may be slightly lower; > 0.95 is fine).

In [ ]:
from google.colab import files
files.download('model/weights/agrismart_export.zip')